# Lejanter Evaluation Interface - Colab

Bu notebook kolay test sureci icindir. Egitim/model uretme akisini degistirmez.

Arayuz sunlari yapar:

- Kullanici bir cephe gorseli yukler.
- YOLO v1/v2/v3 ve hibrit YOLO+SAM2 sonucunu uretir.
- Istenirse VLM image-only ve detection-assisted rapor uretir.
- Sonuc gorsellerini ve raporlari gosterir.
- Puan/yorumlari Drive icindeki SQLite DB'ye kaydeder.


## 1. Repo ve Paketler

In [ ]:
from pathlib import Path
PROJECT_DIR = Path('/content/lejanter_doga_vlm_codex')
%cd /content
!rm -rf /content/lejanter_doga_vlm_codex
!git clone https://github.com/doganalci/lejanter_doga_vlm_codex.git /content/lejanter_doga_vlm_codex
%cd /content/lejanter_doga_vlm_codex
!pip install -q -r requirements.txt
!pip install -q gradio gdown

## 2. Drive ve Model Agirliklari

Weights klasoru Drive linkinden indirilir. Klasor download bos kalirsa bilinen Drive file ID'leriyle fallback yapilir.

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive')

PROJECT_DIR = Path('/content/lejanter_doga_vlm_codex')
REPORTS_DIR = Path('/content/drive/MyDrive/_doganalci_onedrive/codes_doga_doktora_2025/lejant_vllm_sehemntaton_report_may_2026/reports')
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

WEIGHTS_DRIVE_FOLDER_URL = 'https://drive.google.com/drive/folders/1vmSSPsnu4fVBkM0yn1ScU7DFgDXyOksa?usp=sharing'
WEIGHTS_DOWNLOAD_DIR = Path('/content/shared_drive_weights')
if WEIGHTS_DOWNLOAD_DIR.exists():
    shutil.rmtree(WEIGHTS_DOWNLOAD_DIR)
WEIGHTS_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

!gdown --folder '{WEIGHTS_DRIVE_FOLDER_URL}' -O /content/shared_drive_weights --remaining-ok

WEIGHT_FILE_IDS = {
    'elements-seg-v1-best.pt': '12GNVSipuIoK2BwVXx7UWMwpPNnh2Ap4A',
    'elements-seg-v2-aug-controlled-best.pt': '1vuSckiDRJxWLSWgNlWpEhJmX1mnPunM7',
    'elements-seg-v3-no-erasing-best.pt': '1OGtVBhNVBgJq1r0aQc5Eiu5wPz1Wd7ME',
}
if not list(WEIGHTS_DOWNLOAD_DIR.rglob('*.pt')):
    print('No .pt found from folder download; trying direct file IDs...')
    for filename, file_id in WEIGHT_FILE_IDS.items():
        !gdown --id {file_id} -O /content/shared_drive_weights/{filename}

weights_dst = PROJECT_DIR / 'drive_weights'
if weights_dst.exists():
    shutil.rmtree(weights_dst)
weights_dst.mkdir(parents=True, exist_ok=True)
for pt in sorted(WEIGHTS_DOWNLOAD_DIR.rglob('*.pt')):
    shutil.copy2(pt, weights_dst / pt.name)

print('Weights copied to:', weights_dst)
for pt in sorted(weights_dst.rglob('*.pt')):
    print('-', pt)
print('Reports dir:', REPORTS_DIR)

## 3. SAM2 Kurulumu

Hibrit YOLO+SAM2 sonucunu almak icin gerekli. Sadece YOLO ve VLM denemek istersen bu hucreyi atlayabilirsin; arayuz SAM2 yoksa filtrelenmis YOLO sonucuyla devam eder.

In [ ]:
%cd /content
!rm -rf /content/sam2
!git clone https://github.com/facebookresearch/sam2.git /content/sam2
%cd /content/sam2
!pip install -q -e .
!mkdir -p /content/sam2/checkpoints
!wget -q -nc -O /content/sam2/checkpoints/sam2.1_hiera_tiny.pt https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_tiny.pt
%cd /content/lejanter_doga_vlm_codex
print('SAM2 ready')

## 4. Arayuzu Baslat

Cikan Gradio linkini ac. Resim yukle, calisacak deneyleri sec, sonucu gor. Puan/yorum kayitlari Drive'daki SQLite DB'ye yazilir.

In [ ]:
REPORTS_DIR = '/content/drive/MyDrive/_doganalci_onedrive/codes_doga_doktora_2025/lejant_vllm_sehemntaton_report_may_2026/reports'
!cd /content/lejanter_doga_vlm_codex && python scripts/evaluation_interface.py   --project-dir /content/lejanter_doga_vlm_codex   --reports-dir "$REPORTS_DIR"   --weights-dir /content/lejanter_doga_vlm_codex/drive_weights   --db-path "$REPORTS_DIR/evaluation_sessions/ratings.sqlite"   --sam2-dir /content/sam2   --sam2-checkpoint /content/sam2/checkpoints/sam2.1_hiera_tiny.pt   --sam2-model-cfg configs/sam2.1/sam2.1_hiera_t.yaml   --device 0   --share

## 5. Kayitlari Kontrol Et

In [ ]:
from pathlib import Path
REPORTS_DIR = Path('/content/drive/MyDrive/_doganalci_onedrive/codes_doga_doktora_2025/lejant_vllm_sehemntaton_report_may_2026/reports')
!find "$REPORTS_DIR/evaluation_sessions" -maxdepth 3 -type f | sort | sed -n '1,200p'